In [3]:
import pandas as pd
import numpy as np
import os
import glob
from collections import Counter
# ==========================================
# BƯỚC 1 & 2: ĐỌC VÀ GỘP FILE CSV
# ==========================================
folder_path = "./CIC_2017" 
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

print(f"Tìm thấy {len(all_files)} file. Đang tiến hành gộp...")

df_list = []
for file in all_files:
    print(f"Đang đọc: {os.path.basename(file)}")
    df_temp = pd.read_csv(file)
    df_list.append(df_temp)

# Gộp tất cả DataFrames trong list thành một DataFrame tổng
df = pd.concat(df_list, ignore_index=True)
print("\nĐã gộp xong! Tổng số dòng và cột ban đầu:", df.shape)

# ==========================================
# BƯỚC 3: LÀM SẠCH VÀ CHỐNG RÒ RỈ DỮ LIỆU
# ==========================================
# 3.1 Xóa khoảng trắng thừa ở đầu/cuối tên cột
df.columns = df.columns.str.strip()

# 3.2 CHỐNG RÒ RỈ DỮ LIỆU: Lọc và xóa các cột định danh
leakage_cols = [
    'Flow ID', 
    'Source IP', 
    'Src IP',         
    'Destination IP', 
    'Dst IP', 
    'Source Port', 
    'Src Port',
    'Timestamp'
]
cols_to_drop = [col for col in leakage_cols if col in df.columns]
df = df.drop(columns=cols_to_drop)
print(f"Đã xóa {len(cols_to_drop)} cột gây rò rỉ: {cols_to_drop}")

# 3.3 Dọn rác (NaN, Infinity)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print("Số dòng sau khi dọn rác:", df.shape)

# ==========================================
# BƯỚC 4 & 5: TÁCH ĐẶC TRƯNG VÀ NHÃN
# ==========================================
X_raw = df.drop(columns=['Label'])
Y_raw = df['Label']

# Chuyển đổi nhãn (Mã hóa Binary: 0 cho Bình thường, 1 cho Tấn công)
Y = Y_raw.apply(lambda x: 0 if x == 'BENIGN' else 1).values
X = X_raw.values

print("\n--- KẾT QUẢ TIỀN XỬ LÝ TỔNG ---")
print("Kích thước ma trận X:", X.shape)
print("Kích thước mảng Y:", Y.shape)
print("Số cột (đặc trưng) thực tế dùng để huấn luyện:", X.shape[1])
print("Thống kê nhãn Y:", dict(Counter(Y)))

Tìm thấy 8 file. Đang tiến hành gộp...
Đang đọc: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Đang đọc: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Đang đọc: Friday-WorkingHours-Morning.pcap_ISCX.csv
Đang đọc: Monday-WorkingHours.pcap_ISCX.csv
Đang đọc: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Đang đọc: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Đang đọc: Tuesday-WorkingHours.pcap_ISCX.csv
Đang đọc: Wednesday-workingHours.pcap_ISCX.csv

Đã gộp xong! Tổng số dòng và cột ban đầu: (2830743, 79)
Đã xóa 0 cột gây rò rỉ: []
Số dòng sau khi dọn rác: (2827876, 79)

--- KẾT QUẢ TIỀN XỬ LÝ TỔNG ---
Kích thước ma trận X: (2827876, 78)
Kích thước mảng Y: (2827876,)
Số cột (đặc trưng) thực tế dùng để huấn luyện: 78
Thống kê nhãn Y: {np.int64(0): 2271320, np.int64(1): 556556}


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

print("--- BẮT ĐẦU HUẤN LUYỆN HỌC TẬP TRUNG (CENTRALIZED LEARNING) ---")
start_time = time.time()

# 1. Chia dữ liệu thành tập huấn luyện (80%) và tập kiểm tra (20%)
X_train_cen, X_test_cen, y_train_cen, y_test_cen = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train_cen.shape[0]} mẫu")
print(f"Kích thước tập Test: {X_test_cen.shape[0]} mẫu")

# 2. Khởi tạo mô hình Random Forest
centralized_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)

# 3. Huấn luyện
print("\nĐang trồng Khu rừng 200 cây trên toàn bộ dữ liệu (Vui lòng đợi vài phút)...")
centralized_model.fit(X_train_cen, y_train_cen)

# 4. Dự đoán và Chấm điểm
print("Đang đánh giá mô hình...")
y_pred_cen = centralized_model.predict(X_test_cen)

acc_cen = accuracy_score(y_test_cen, y_pred_cen)
prec_cen = precision_score(y_test_cen, y_pred_cen, average='binary', zero_division=0)

# ✅ ĐÃ SỬA TẠI ĐÂY: Dùng hàm recall_score thay cho precision_score
rec_cen = recall_score(y_test_cen, y_pred_cen, average='binary', zero_division=0) 

f1_cen = f1_score(y_test_cen, y_pred_cen, average='binary', zero_division=0)

end_time = time.time()

# 5. In kết quả
print(f"\n{'='*50}")
print("[KẾT QUẢ HỌC TẬP TRUNG (BASELINE)]")
print(f"{'='*50}")
print(f"Kích thước rừng : {centralized_model.n_estimators} cây")
print(f"Độ chính xác    : {acc_cen * 100:.4f}%")
print(f"Độ chuẩn (Prec) : {prec_cen * 100:.4f}%")
print(f"Độ nhạy (Recall): {rec_cen * 100:.4f}%")
print(f"Điểm F1-Score   : {f1_cen * 100:.4f}%")
print(f"Thời gian chạy  : {end_time - start_time:.2f} giây")
print(f"{'='*50}\n")

--- BẮT ĐẦU HUẤN LUYỆN HỌC TẬP TRUNG (CENTRALIZED LEARNING) ---
Kích thước tập Train: 2262300 mẫu
Kích thước tập Test: 565576 mẫu

Đang trồng Khu rừng 200 cây trên toàn bộ dữ liệu (Vui lòng đợi vài phút)...
Đang đánh giá mô hình...

[KẾT QUẢ HỌC TẬP TRUNG (BASELINE)]
Kích thước rừng : 200 cây
Độ chính xác    : 99.8863%
Độ chuẩn (Prec) : 99.7308%
Độ nhạy (Recall): 99.6923%
Điểm F1-Score   : 99.7116%
Thời gian chạy  : 676.56 giây



In [ ]:
import os
from sklearn.model_selection import train_test_split

# 1. Tạo thư mục để chứa "bài đã chia"
out_dir = "./client_data"
os.makedirs(out_dir, exist_ok=True)

print(f"Đang chia tách 2.8 triệu dòng dữ liệu...")

# 2. Tách 20% dữ liệu làm tập Test chung cho Server
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

# 3. Chia đều 80% dữ liệu Train cho 20 Client
num_clients = 20
X_splits = np.array_split(X_train, num_clients)
y_splits = np.array_split(y_train, num_clients)

# 4. Lưu dữ liệu từng Client ra ổ cứng bằng định dạng nén tối ưu của Numpy
for i in range(num_clients):
    np.savez_compressed(f"{out_dir}/client_{i}.npz", X=X_splits[i], Y=y_splits[i])
    if i < 3 or i == num_clients - 1: # Chỉ in vài client để đỡ dài dòng
        print(f"Đã đóng gói dữ liệu cho Client {i}: {len(y_splits[i])} mẫu.")

# 5. Lưu riêng tập Test cho Server
np.savez_compressed(f"{out_dir}/server_test.npz", X=X_test, Y=y_test)
print(f"Đã đóng gói dữ liệu Test cho Server: {len(y_test)} mẫu.")
print(f"Hoàn tất! Các file siêu nhẹ đã nằm sẵn trong thư mục {out_dir}")

Đang chia tách 2.8 triệu dòng dữ liệu...
Đã đóng gói dữ liệu cho Client 0: 113115 mẫu.
Đã đóng gói dữ liệu cho Client 1: 113115 mẫu.
Đã đóng gói dữ liệu cho Client 2: 113115 mẫu.
Đã đóng gói dữ liệu cho Client 19: 113115 mẫu.
Đã đóng gói dữ liệu Test cho Server: 565576 mẫu.
Hoàn tất! Các file siêu nhẹ đã nằm sẵn trong thư mục ./client_data


: 

In [5]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

out_dir_non_iid = "./client_data_non_iid"
os.makedirs(out_dir_non_iid, exist_ok=True)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

def dirichlet_non_iid_split(X, y, num_clients=20, alpha=0.5, min_per_class=200, seed=42):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    client_idx = [[] for _ in range(num_clients)]

    for c in classes:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)

        # Bước 1: trích "sàn an toàn" — đảm bảo MỌI Client có tối thiểu min_per_class mẫu của lớp này
        guaranteed_n = min(min_per_class * num_clients, len(idx_c))
        reserved, remaining = idx_c[:guaranteed_n], idx_c[guaranteed_n:]
        reserved_splits = np.array_split(reserved, num_clients)

        # Bước 2: phần còn lại chia theo tỉ lệ Dirichlet — tạo độ lệch Non-IID thật sự
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        cuts = (np.cumsum(proportions) * len(remaining)).astype(int)[:-1]
        remaining_splits = np.split(remaining, cuts)

        for i in range(num_clients):
            client_idx[i].extend(reserved_splits[i])
            client_idx[i].extend(remaining_splits[i])

    return [np.array(idx) for idx in client_idx]

num_clients = 20
client_indices = dirichlet_non_iid_split(X_train, y_train, num_clients=num_clients, alpha=0.5, min_per_class=200)

print(f"Thống kê nhãn của từng Client (Lưu tại {out_dir_non_iid}):")
for i, idx in enumerate(client_indices):
    X_i, y_i = X_train[idx], y_train[idx]
    np.savez_compressed(f"{out_dir_non_iid}/client_{i}.npz", X=X_i, Y=y_i)
    print(f"Client {i:2d}: {dict(Counter(y_i))}")

np.savez_compressed(f"{out_dir_non_iid}/server_test.npz", X=X_test, Y=y_test)
print(f"\nĐã đóng gói dữ liệu Test cho Server: {len(y_test)} mẫu.")

Thống kê nhãn của từng Client (Lưu tại ./client_data_non_iid):
Client  0: {np.int64(0): 89254, np.int64(1): 21829}
Client  1: {np.int64(0): 106769, np.int64(1): 62215}
Client  2: {np.int64(0): 190746, np.int64(1): 53446}
Client  3: {np.int64(0): 23348, np.int64(1): 57878}
Client  4: {np.int64(0): 28699, np.int64(1): 215}
Client  5: {np.int64(0): 12892, np.int64(1): 201}
Client  6: {np.int64(0): 23897, np.int64(1): 3341}
Client  7: {np.int64(0): 89174, np.int64(1): 100881}
Client  8: {np.int64(0): 73177, np.int64(1): 17498}
Client  9: {np.int64(0): 26107, np.int64(1): 6741}
Client 10: {np.int64(0): 76160, np.int64(1): 46635}
Client 11: {np.int64(0): 172305, np.int64(1): 28184}
Client 12: {np.int64(0): 67151, np.int64(1): 200}
Client 13: {np.int64(0): 36763, np.int64(1): 3936}
Client 14: {np.int64(0): 130001, np.int64(1): 4400}
Client 15: {np.int64(0): 248090, np.int64(1): 21509}
Client 16: {np.int64(0): 21645, np.int64(1): 228}
Client 17: {np.int64(0): 63089, np.int64(1): 979}
Client 18

In [1]:
import os
import glob

print("--- ĐÁNH GIÁ CHI PHÍ GIAO TIẾP MẠNG (COMMUNICATION COST) ---\n")

# 1. Chi phí mạng của Học Tập Trung (Gửi toàn bộ Dữ liệu thô)
client_dir = "./client_data_non_iid"
raw_files = glob.glob(f"{client_dir}/client_*.npz")

total_raw_size_bytes = sum(os.path.getsize(f) for f in raw_files)
total_raw_size_mb = total_raw_size_bytes / (1024 * 1024)

print(f"[1] Kịch bản Học Tập Trung (Centralized Learning):")
print(f"    - Số lượng file phải truyền : {len(raw_files)} file dữ liệu thô")
print(f"    - Tổng chi phí băng thông   : {total_raw_size_mb:.2f} MB\n")

# 2. Chi phí mạng của Học Liên Kết (Chỉ gửi Tham số mô hình)
# Tổng dữ liệu 20 Client đẩy lên Server chính là 200 cây quyết định được đóng gói
model_file = "ids_global_rf_model.pkl"

if os.path.exists(model_file):
    model_size_bytes = os.path.getsize(model_file)
    model_size_mb = model_size_bytes / (1024 * 1024)
    
    print(f"[2] Kịch bản Học Liên Kết (Federated Learning - Vòng 1):")
    print(f"    - Payload (20 Client -> Server): 20x10 = 200 cây quyết định")
    print(f"    - Tổng chi phí băng thông      : {model_size_mb:.2f} MB\n")
    
    # 3. Tính toán tỷ lệ tiết kiệm
    savings = total_raw_size_mb / model_size_mb
    reduction_percent = ((total_raw_size_mb - model_size_mb) / total_raw_size_mb) * 100
    
    print(f"{'='*50}")
    print("[KẾT LUẬN TỐI ƯU BĂNG THÔNG]")
    print(f"{'='*50}")
    print(f"Federated Learning giúp tiết kiệm {reduction_percent:.2f}% băng thông mạng.")
    print(f"Nhẹ hơn mô hình Tập trung gấp {savings:.1f} lần!")
    print(f"{'='*50}\n")
else:
    print(f"Không tìm thấy file {model_file}. Hãy đảm bảo bạn chạy file này ở cùng thư mục chứa mô hình đã lưu.")

--- ĐÁNH GIÁ CHI PHÍ GIAO TIẾP MẠNG (COMMUNICATION COST) ---

[1] Kịch bản Học Tập Trung (Centralized Learning):
    - Số lượng file phải truyền : 20 file dữ liệu thô
    - Tổng chi phí băng thông   : 267.85 MB

[2] Kịch bản Học Liên Kết (Federated Learning - Vòng 1):
    - Payload (20 Client -> Server): 20x10 = 200 cây quyết định
    - Tổng chi phí băng thông      : 11.86 MB

[KẾT LUẬN TỐI ƯU BĂNG THÔNG]
Federated Learning giúp tiết kiệm 95.57% băng thông mạng.
Nhẹ hơn mô hình Tập trung gấp 22.6 lần!



In [1]:
import numpy as np, glob
from collections import Counter

for folder in ["client_data", "client_data_non_iid"]:
    print(f"--- {folder} ---")
    for f in sorted(glob.glob(f"./{folder}/client_*.npz")):
        d = np.load(f)
        print(f, dict(Counter(d["Y"])))

--- client_data ---
./client_data\client_0.npz {np.int64(0): 90735, np.int64(1): 22380}
./client_data\client_1.npz {np.int64(0): 90929, np.int64(1): 22186}
./client_data\client_10.npz {np.int64(0): 90950, np.int64(1): 22165}
./client_data\client_11.npz {np.int64(0): 90630, np.int64(1): 22485}
./client_data\client_12.npz {np.int64(0): 90769, np.int64(1): 22346}
./client_data\client_13.npz {np.int64(0): 90900, np.int64(1): 22215}
./client_data\client_14.npz {np.int64(0): 90896, np.int64(1): 22219}
./client_data\client_15.npz {np.int64(0): 90644, np.int64(1): 22471}
./client_data\client_16.npz {np.int64(0): 90897, np.int64(1): 22218}
./client_data\client_17.npz {np.int64(0): 90894, np.int64(1): 22221}
./client_data\client_18.npz {np.int64(0): 90937, np.int64(1): 22178}
./client_data\client_19.npz {np.int64(0): 90951, np.int64(1): 22164}
./client_data\client_2.npz {np.int64(0): 90791, np.int64(1): 22324}
./client_data\client_3.npz {np.int64(0): 90772, np.int64(1): 22343}
./client_data\clie